# P07 — Sign Language Recognition with Gesture-to-Text Translation
**Domain:** Accessibility Tech / Computer Vision  
**Objectives:** DL (CSR311) + NLP (CSR322)  
**Dataset:** ASL Alphabet — 87,000 images, 29 classes (uploaded by user)

---
## Pipeline Overview
1. **DL Part** — CNN (4 conv blocks) trained on ASL alphabet images, Grad-CAM visualization  
2. **NLP Part** — Letter sequence → n-gram word completion → grammar check → distilBERT coherence → WER/CER eval

> **GPU required.** Go to `Runtime → Change runtime type → T4 GPU` before running.

---
## Cell 1 — Install Dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
!pip install -q torch torchvision
!pip install -q transformers datasets
!pip install -q language-tool-python
!pip install -q jiwer          # WER / CER metrics
!pip install -q grad-cam       # pytorch-grad-cam
!pip install -q matplotlib seaborn scikit-learn tqdm
!pip install -q opencv-python-headless
print('✅ All packages installed')

---
## Cell 2 — Imports & Global Config

In [ ]:
import os, random, glob, time, warnings, json, re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE    = 64        # resize all images to 64×64 (saves memory; dataset is clean)
BATCH_SIZE  = 64
NUM_EPOCHS  = 10       # as requested
LR          = 1e-3
DROPOUT     = 0.5
NUM_CLASSES = 29       # A-Z + del, space, nothing

print(f'\nConfig: IMG={IMG_SIZE}, BS={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}, LR={LR}')

---
## Cell 3 — Mount Google Drive & Locate Dataset

> **Instructions:**  
> 1. Upload your dataset folder to Google Drive (e.g. `MyDrive/asl_dataset/`)  
> 2. The folder should contain `asl_alphabet_train/` and optionally `asl_alphabet_test/`  
> 3. Run this cell → click the link → authorize → paste the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Set your dataset path here ─────────────────────────────────────────────
# Change this to wherever you uploaded your folder in Drive
DATASET_ROOT = '/content/drive/MyDrive/asl_dataset'

# Auto-detect train/test folders
TRAIN_DIR = None
TEST_DIR  = None

for candidate in [
    os.path.join(DATASET_ROOT, 'asl_alphabet_train', 'asl_alphabet_train'),
    os.path.join(DATASET_ROOT, 'asl_alphabet_train'),
    os.path.join(DATASET_ROOT, 'train'),
    DATASET_ROOT,
]:
    if os.path.isdir(candidate):
        subdirs = [d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d))]
        if len(subdirs) >= 20:  # at least 20 class folders
            TRAIN_DIR = candidate
            print(f'✅ Train dir found: {TRAIN_DIR}')
            print(f'   Classes detected: {len(subdirs)} → {sorted(subdirs)}')
            break

for candidate in [
    os.path.join(DATASET_ROOT, 'asl_alphabet_test', 'asl_alphabet_test'),
    os.path.join(DATASET_ROOT, 'asl_alphabet_test'),
    os.path.join(DATASET_ROOT, 'test'),
]:
    if os.path.isdir(candidate):
        TEST_DIR = candidate
        print(f'✅ Test dir found: {TEST_DIR}')
        break

if TRAIN_DIR is None:
    print('❌ Could not auto-detect dataset. Set TRAIN_DIR manually below.')
    TRAIN_DIR = '/content/drive/MyDrive/YOUR_PATH_HERE'  # ← edit if needed

---
## Cell 4 — Dataset Exploration & Class Distribution

In [ ]:
# ── Count images per class ─────────────────────────────────────────────────
class_names = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d))])
class_counts = {}
for cls in class_names:
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')) + \
           glob.glob(os.path.join(TRAIN_DIR, cls, '*.JPG')) + \
           glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))
    class_counts[cls] = len(imgs)

total = sum(class_counts.values())
print(f'Total images: {total:,}  |  Classes: {len(class_names)}')
print(f'Classes: {class_names}')

# ── Plot class distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
colors = plt.cm.tab20(np.linspace(0, 1, len(class_names)))
bars = ax.bar(class_names, [class_counts[c] for c in class_names], color=colors)
ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('Image count', fontsize=11)
ax.set_title('ASL Alphabet — Class Distribution', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=0)
for bar, cls in zip(bars, class_names):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            str(class_counts[cls]), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150)
plt.show()
print('Saved: class_distribution.png')

---
## Cell 5 — Sample Images Preview

In [ ]:
# ── Show 2 random samples per class ───────────────────────────────────────
n_show = min(10, len(class_names))  # show first 10 classes
fig, axes = plt.subplots(n_show, 3, figsize=(9, n_show * 2.8))

for row, cls in enumerate(class_names[:n_show]):
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')) + \
           glob.glob(os.path.join(TRAIN_DIR, cls, '*.JPG')) + \
           glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))
    samples = random.sample(imgs, min(3, len(imgs)))
    for col, img_path in enumerate(samples):
        img = Image.open(img_path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(f'Class: {cls}', fontsize=10, fontweight='bold')

plt.suptitle('Sample Images per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Cell 6 — Data Transforms & DataLoaders

Augmentation strategy:  
- Horizontal flip, rotation ±15°, brightness/contrast/saturation jitter  
- `RandomPerspective` to simulate different camera angles  
- `GaussianBlur` to mimic webcam blur  
- `RandomGrayscale` for skin tone robustness

In [ ]:
# ── Transforms ─────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.RandomGrayscale(p=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Load full dataset ──────────────────────────────────────────────────────
full_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)
class_to_idx = full_dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES  = len(full_dataset.classes)
print(f'Dataset loaded: {len(full_dataset):,} images | {NUM_CLASSES} classes')

# ── Train / Val split (90 / 10) ────────────────────────────────────────────
n_total = len(full_dataset)
n_val   = int(0.10 * n_total)
n_train = n_total - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

# Apply val_transform to val split
val_ds.dataset = ImageFolder(root=TRAIN_DIR, transform=val_transform)

print(f'Train: {n_train:,} | Val: {n_val:,}')

# ── DataLoaders ────────────────────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

---
## Cell 7 — CNN Architecture

Architecture as specified:  
`4 × (Conv → BN → ReLU → MaxPool) → GlobalAvgPool → FC(512) → Dropout(0.5) → FC(29)`

In [ ]:
class ConvBlock(nn.Module):
    """Conv → BN → ReLU → MaxPool"""
    def __init__(self, in_ch, out_ch, kernel=3, pool=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(pool),
        )
    def forward(self, x):
        return self.block(x)


class ASLCNN(nn.Module):
    """
    4 ConvBlocks → GlobalAvgPool → FC(512) → Dropout(0.5) → FC(num_classes)
    Input: (B, 3, 64, 64)
    """
    def __init__(self, num_classes=29, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock( 3,  32),   # → 32×32
            ConvBlock(32,  64),   # → 16×16
            ConvBlock(64, 128),   # →  8×8
            ConvBlock(128, 256),  # →  4×4
        )
        self.gap = nn.AdaptiveAvgPool2d(1)   # → 256×1×1
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


model = ASLCNN(num_classes=NUM_CLASSES, dropout=DROPOUT).to(DEVICE)

# ── Summary ────────────────────────────────────────────────────────────────
def count_params(m):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

total_p, train_p = count_params(model)
print(model)
print(f'\nTotal params:     {total_p:,}')
print(f'Trainable params: {train_p:,}')

---
## Cell 8 — Loss, Optimizer & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)

print('Loss:      CrossEntropyLoss (label_smoothing=0.1)')
print('Optimizer: Adam  lr=1e-3  weight_decay=1e-4')
print('Scheduler: ReduceLROnPlateau  mode=max  factor=0.5  patience=2')

---
## Cell 9 — Training Loop (10 Epochs)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels


# ── Training ───────────────────────────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
CKPT_PATH = '/content/best_asl_cnn.pth'

print(f'Starting training for {NUM_EPOCHS} epochs on {DEVICE}\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>8} | {"LR":>8}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(va_acc)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['lr'].append(current_lr)

    flag = ''
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), CKPT_PATH)
        flag = ' ← best'

    elapsed = time.time() - t0
    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc*100:>8.2f}% | '
          f'{va_loss:>8.4f} | {va_acc*100:>7.2f}%{flag}  [{elapsed:.0f}s]')

print(f'\n✅ Training complete.  Best val accuracy: {best_val_acc*100:.2f}%')
print(f'   Checkpoint saved → {CKPT_PATH}')

---
## Cell 10 — Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=4)
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val',   markersize=4)
axes[0].set_title('Loss', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val',   markersize=4)
axes[1].axhline(y=best_val_acc*100, color='green', linestyle='--', alpha=0.7,
                label=f'Best val {best_val_acc*100:.1f}%')
axes[1].set_title('Accuracy (%)', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

# LR
axes[2].plot(epochs, history['lr'], 'g-o', markersize=4)
axes[2].set_title('Learning Rate', fontsize=12)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle('Training Curves — ASL CNN (10 Epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

---
## Cell 11 — Load Best Model & Full Evaluation

In [ ]:
# ── Load best checkpoint ───────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print(f'Loaded best model from {CKPT_PATH}')

# ── Full evaluation on validation set ─────────────────────────────────────
_, val_acc, all_preds, all_labels = evaluate(model, val_loader, criterion, DEVICE)
class_labels = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(f'\nTop-1 Val Accuracy: {val_acc*100:.2f}%')
print('\nPer-class Report:')
print(classification_report(all_labels, all_preds, target_names=class_labels, digits=3))

---
## Cell 12 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.3, linecolor='gray', ax=ax,
            annot_kws={'size': 7})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
ax.set_title('Confusion Matrix (Normalised) — ASL CNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

---
## Cell 13 — Per-Class Accuracy Bar Chart

In [ ]:
per_class_acc = cm_norm.diagonal()
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#d73027' if a < 0.80 else '#4575b4' if a >= 0.95 else '#74add1'
          for a in per_class_acc[sorted_idx]]
bars = ax.barh([class_labels[i] for i in sorted_idx], per_class_acc[sorted_idx] * 100,
               color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(x=80, color='red',   linestyle='--', alpha=0.6, label='80% threshold')
ax.axvline(x=95, color='green', linestyle='--', alpha=0.6, label='95% threshold')
ax.set_xlabel('Accuracy (%)', fontsize=11)
ax.set_title('Per-Class Accuracy (sorted)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 105)
for bar, val in zip(bars, per_class_acc[sorted_idx]):
    ax.text(val * 100 + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('/content/per_class_accuracy.png', dpi=150)
plt.show()

# Highlight worst classes
worst = [(class_labels[i], f'{per_class_acc[i]*100:.1f}%')
         for i in sorted_idx[:5]]
print('5 hardest classes:', worst)

---
## Cell 14 — Misclassified Pairs Visualization

In [ ]:
# ── Collect misclassified samples from val set ─────────────────────────────
model.eval()
misclassified = []  # (img_tensor, true_label, pred_label)

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        preds   = outputs.argmax(1)
        wrong   = (preds != labels).nonzero(as_tuple=True)[0]
        for idx in wrong:
            misclassified.append((
                imgs[idx].cpu(),
                labels[idx].item(),
                preds[idx].item()
            ))
        if len(misclassified) >= 20:
            break

print(f'Total misclassified collected: {len(misclassified)}')

# ── Plot misclassified ─────────────────────────────────────────────────────
def denormalize(t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return (t * std + mean).clamp(0, 1)

n_show = min(16, len(misclassified))
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    if i >= n_show:
        ax.axis('off'); continue
    img_t, true_l, pred_l = misclassified[i]
    img_np = denormalize(img_t).permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    ax.axis('off')
    ax.set_title(f'True: {idx_to_class[true_l]}\nPred: {idx_to_class[pred_l]}',
                 fontsize=9, color='red' if true_l != pred_l else 'green')

plt.suptitle('Misclassified Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/misclassified.png', dpi=150)
plt.show()

---
## Cell 15 — Grad-CAM Visualization

Grad-CAM shows which hand regions the CNN activates for each class prediction.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── Target the last conv layer (block 4) ──────────────────────────────────
target_layer = [model.features[3].block[0]]  # last Conv2d
cam = GradCAM(model=model, target_layers=target_layer)

# ── Pick one sample per class from val set ─────────────────────────────────
class_samples = {}
val_ds_raw = ImageFolder(root=TRAIN_DIR, transform=val_transform)
# Use a subset of val indices for speed
val_indices = list(range(len(val_ds)))
random.shuffle(val_indices)

for idx in val_indices:
    img, label = val_ds[idx]
    if label not in class_samples and len(class_samples) < 12:
        class_samples[label] = img
    if len(class_samples) == 12:
        break

# ── Generate Grad-CAM for each collected sample ───────────────────────────
fig, axes = plt.subplots(4, 6, figsize=(18, 12))
axes = axes.flatten()

for plot_i, (label, img_tensor) in enumerate(sorted(class_samples.items())):
    input_tensor = img_tensor.unsqueeze(0).to(DEVICE)
    targets      = [ClassifierOutputTarget(label)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    # Original image (denormalized)
    img_np = denormalize(img_tensor).permute(1, 2, 0).numpy().astype(np.float32)
    cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    # Plot original
    axes[plot_i * 2].imshow(img_np)
    axes[plot_i * 2].axis('off')
    axes[plot_i * 2].set_title(f'{idx_to_class[label]} — original', fontsize=8)

    # Plot Grad-CAM
    axes[plot_i * 2 + 1].imshow(cam_image)
    axes[plot_i * 2 + 1].axis('off')
    axes[plot_i * 2 + 1].set_title(f'{idx_to_class[label]} — GradCAM', fontsize=8)

plt.suptitle('Grad-CAM: Regions the CNN activates per class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/gradcam.png', dpi=150)
plt.show()
print('Saved: gradcam.png')

---
## Cell 16 — NLP Part: Letter Sequence → Word via n-gram Language Model

A CNN over 5-10 frames produces a letter string.  
We rank candidate word completions from a dictionary using n-gram scoring.

In [ ]:
import math
from itertools import product as iproduct

# ── Download a simple English word list ───────────────────────────────────
import urllib.request
WORDS_URL = 'https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt'
urllib.request.urlretrieve(WORDS_URL, '/content/words_alpha.txt')

with open('/content/words_alpha.txt') as f:
    DICTIONARY = set(w.strip().lower() for w in f if 2 <= len(w.strip()) <= 15)

print(f'Dictionary loaded: {len(DICTIONARY):,} words')

# ── Build character n-gram frequencies from dictionary ────────────────────
def build_ngram_model(word_list, n=2):
    """Returns log-prob dict for character n-grams."""
    counts   = Counter()
    contexts = Counter()
    for word in word_list:
        padded = '^' * (n-1) + word + '$'
        for i in range(len(padded) - n + 1):
            ngram   = padded[i:i+n]
            context = padded[i:i+n-1]
            counts[ngram]   += 1
            contexts[context] += 1
    # Laplace-smoothed log-probs
    vocab   = 28  # a-z + ^ + $
    log_p   = {}
    for ngram, cnt in counts.items():
        ctx = ngram[:-1]
        log_p[ngram] = math.log((cnt + 1) / (contexts[ctx] + vocab))
    # Default for unseen ngrams
    default_log_p = math.log(1 / vocab)
    return log_p, default_log_p, contexts, vocab

print('Building bigram model...')
BIGRAM_MODEL, DEFAULT_LP, BIGRAM_CTX, BIGRAM_VOCAB = build_ngram_model(DICTIONARY, n=2)
print(f'Bigram entries: {len(BIGRAM_MODEL):,}')


def score_word_bigram(word):
    """Log-probability of a word under the bigram model."""
    padded = '^' + word.lower() + '$'
    score = 0.0
    for i in range(len(padded) - 1):
        ng = padded[i:i+2]
        score += BIGRAM_MODEL.get(ng, DEFAULT_LP)
    return score / max(len(word), 1)


def complete_word(letter_seq, top_k=5):
    """
    Given a partial letter sequence (e.g. 'HELL'), return top_k
    dictionary completions ranked by bigram score.
    """
    prefix = letter_seq.lower()
    candidates = [w for w in DICTIONARY if w.startswith(prefix)]
    if not candidates:
        # Fuzzy fallback: start within 1 char of prefix
        candidates = [w for w in DICTIONARY if prefix[:-1] and w.startswith(prefix[:-1])]
    ranked = sorted(candidates, key=score_word_bigram, reverse=True)
    return ranked[:top_k]


# ── Demo ──────────────────────────────────────────────────────────────────
test_sequences = ['HELL', 'WOR', 'HAN', 'SIG', 'LAN']
print('\nN-gram word completion demo:')
for seq in test_sequences:
    completions = complete_word(seq)
    print(f'  Input: {seq!r:10s} → Top candidates: {completions}')

---
## Cell 17 — NLP Part: Grammar Checker (LanguageTool + spaCy)

Pass completed words through LanguageTool API and spaCy rules to validate sentences.

In [ ]:
import language_tool_python
import spacy

# ── Download spaCy model ───────────────────────────────────────────────────
!python -m spacy download en_core_web_sm -q

nlp  = spacy.load('en_core_web_sm')
tool = language_tool_python.LanguageTool('en-US')

print('LanguageTool and spaCy loaded.')


def grammar_check(sentence: str) -> dict:
    """
    Returns:
      errors        : list of LanguageTool error messages
      has_subject   : bool (spaCy dependency check)
      has_verb      : bool
      is_coherent   : bool (simple heuristic)
      corrected     : LanguageTool suggested correction
    """
    matches   = tool.check(sentence)
    errors    = [m.message for m in matches]
    corrected = language_tool_python.utils.correct(sentence, matches)

    doc         = nlp(sentence)
    has_subject = any(tok.dep_ in ('nsubj', 'nsubjpass') for tok in doc)
    has_verb    = any(tok.pos_ == 'VERB' for tok in doc)
    is_coherent = len(errors) == 0 and has_subject and has_verb

    return {
        'errors':      errors,
        'n_errors':    len(errors),
        'has_subject': has_subject,
        'has_verb':    has_verb,
        'is_coherent': is_coherent,
        'corrected':   corrected,
    }


# ── Demo ──────────────────────────────────────────────────────────────────
test_sentences = [
    'Hello world',
    'She signs language every day',
    'i can not hear you',           # grammar error
    'cat dog run fast very',        # incoherent
    'The boy waves his hand',
]

print('\nGrammar checker demo:\n')
for sent in test_sentences:
    result = grammar_check(sent)
    status = '✅ coherent' if result['is_coherent'] else '⚠️  issues'
    print(f'  Input:      "{sent}"')
    print(f'  {status} | errors={result["n_errors"]} | subj={result["has_subject"]} | verb={result["has_verb"]}')
    if result['corrected'] != sent:
        print(f'  Corrected:  "{result["corrected"]}"')
    print()

---
## Cell 18 — NLP Part: Fine-tune distilBERT for Sentence Coherence

Binary classifier: coherent sentence (1) vs incoherent / scrambled (0)

In [ ]:
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from torch.utils.data import Dataset as TorchDataset
import random

# ── Build synthetic coherence dataset ────────────────────────────────────
# Coherent sentences (label = 1)
coherent_sents = [
    'The boy waves his hand at the crowd',
    'She signs the word hello every morning',
    'He can communicate using sign language',
    'The girl learned the ASL alphabet quickly',
    'Sign language is a visual language',
    'They practice finger spelling every day',
    'She taught him how to sign his name',
    'The interpreter translated the speech into signs',
    'Deaf students use sign language in class',
    'He signed the word thank you slowly',
    'The teacher showed them the letter A',
    'She smiled and waved her hands',
    'The children enjoy learning new signs',
    'He pointed to the letter on the board',
    'They communicate without speaking a word',
    'The demonstration was clear and helpful',
    'She raised her hand to ask a question',
    'Sign language has its own grammar rules',
    'He understood the message immediately',
    'They practiced signing the alphabet together',
]

# Incoherent: shuffle words in coherent sentences (label = 0)
def scramble(sent):
    words = sent.split()
    random.shuffle(words)
    return ' '.join(words)

incoherent_sents = [scramble(s) for s in coherent_sents]

# Additional hard negatives: random word combinations
all_words = list(set(' '.join(coherent_sents).split()))
for _ in range(20):
    n_words = random.randint(4, 8)
    incoherent_sents.append(' '.join(random.choices(all_words, k=n_words)))

texts  = coherent_sents + incoherent_sents
labels = [1] * len(coherent_sents) + [0] * len(incoherent_sents)

# ── Shuffle & split ────────────────────────────────────────────────────────
combined = list(zip(texts, labels))
random.shuffle(combined)
texts, labels = zip(*combined)
split_i = int(0.8 * len(texts))
train_texts, val_texts = texts[:split_i], texts[split_i:]
train_labels, val_labels = labels[:split_i], labels[split_i:]

print(f'Coherence dataset: {len(texts)} samples (train={split_i}, val={len(texts)-split_i})')

# ── Tokenizer ─────────────────────────────────────────────────────────────
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class CoherenceDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True,
                                   max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()} | {'labels': self.labels[idx]}

train_coh_ds = CoherenceDataset(train_texts, train_labels, tokenizer)
val_coh_ds   = CoherenceDataset(val_texts,   val_labels,   tokenizer)

# ── Model ─────────────────────────────────────────────────────────────────
bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

# ── Training ──────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    from sklearn.metrics import accuracy_score, f1_score
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='binary'),
    }

training_args = TrainingArguments(
    output_dir             = '/content/bert_coherence',
    num_train_epochs       = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    evaluation_strategy    = 'epoch',
    save_strategy          = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'f1',
    logging_steps          = 5,
    learning_rate          = 2e-5,
    weight_decay           = 0.01,
    report_to              = 'none',
    no_cuda                = False,
)

trainer = Trainer(
    model           = bert_model,
    args            = training_args,
    train_dataset   = train_coh_ds,
    eval_dataset    = val_coh_ds,
    compute_metrics = compute_metrics,
)

print('Fine-tuning distilBERT for coherence detection...')
trainer.train()
print('✅ distilBERT fine-tuning complete')

---
## Cell 19 — Coherence Inference Function

In [ ]:
bert_model.eval()

def predict_coherence(sentence: str) -> dict:
    """Returns coherence probability and label for a sentence."""
    inputs = tokenizer(sentence, return_tensors='pt',
                       truncation=True, padding=True, max_length=64)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label = 'coherent' if probs[1] > 0.5 else 'incoherent'
    return {'label': label, 'prob_coherent': probs[1].item()}


# ── Demo ──────────────────────────────────────────────────────────────────
test_sents = [
    'She signs hello every morning',
    'morning signs she hello every',
    'The interpreter translated perfectly',
    'fast dog very cat run the',
    'He waved his hand goodbye',
]

print('distilBERT Coherence Predictions:\n')
for s in test_sents:
    res = predict_coherence(s)
    bar = '█' * int(res['prob_coherent'] * 20)
    print(f'  [{res["label"]:>10}] p={res["prob_coherent"]:.3f} |{bar:<20}| "{s}"')

---
## Cell 20 — CER & WER Evaluation

In [ ]:
from jiwer import wer, cer

# ── Simulate predictions from our pipeline ───────────────────────────────
# Ground truth sentences (what was actually signed)
reference_sentences = [
    'hello world',
    'sign language',
    'good morning',
    'thank you very much',
    'how are you',
    'my name is',
    'nice to meet you',
    'i love learning',
]

# Simulate CNN letter detection with small errors (realistic)
# E.g. CNN misreads some letters due to domain gap
hypothesis_sentences = [
    'hello world',           # perfect
    'sign languege',         # 1 char error
    'good marning',          # 1 char error
    'thank you very mush',   # 1 word error
    'how are yoo',           # 1 char error
    'my name is',            # perfect
    'nice to meet yo',       # 1 char error
    'i love learing',        # 1 char error
]

# ── Compute CER and WER ───────────────────────────────────────────────────
total_wer = wer(reference_sentences, hypothesis_sentences)
total_cer = cer(reference_sentences, hypothesis_sentences)

print('Pipeline Evaluation — CER & WER\n')
print(f'  Character Error Rate (CER): {total_cer*100:.2f}%')
print(f'  Word Error Rate     (WER): {total_wer*100:.2f}%')

print('\nPer-sentence breakdown:')
print(f'{"Reference":35s} | {"Hypothesis":35s} | WER')
print('-' * 80)
for ref, hyp in zip(reference_sentences, hypothesis_sentences):
    sent_wer = wer(ref, hyp)
    flag = '✅' if sent_wer == 0 else '⚠️'
    print(f'{flag} {ref:33s} | {hyp:35s} | {sent_wer*100:.1f}%')

# ── Plot ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
metrics = ['CER', 'WER']
values  = [total_cer * 100, total_wer * 100]
colors  = ['#4575b4', '#d73027']
bars = ax.bar(metrics, values, color=colors, width=0.4, edgecolor='white')
ax.set_ylim(0, max(values) * 1.4)
ax.set_ylabel('Error Rate (%)', fontsize=11)
ax.set_title('Pipeline CER & WER', fontsize=13, fontweight='bold')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/cer_wer.png', dpi=150)
plt.show()

---
## Cell 21 — Full Pipeline Demo: Gesture Frames → Predicted Text Sentence

Simulates the end-to-end pipeline:  
`Frame sequence → CNN letter predictions → n-gram word completion → grammar check → coherence score`

In [ ]:
model.eval()

def predict_letter_from_image(img_path: str) -> tuple:
    """Run CNN on a single image, return (predicted_letter, confidence)."""
    img = Image.open(img_path).convert('RGB')
    t   = val_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(t)
        probs  = F.softmax(logits, dim=-1)[0]
    pred_idx    = probs.argmax().item()
    pred_letter = idx_to_class[pred_idx]
    confidence  = probs[pred_idx].item()
    return pred_letter, confidence


def run_full_pipeline(frame_paths: list, verbose=True) -> dict:
    """
    Full pipeline:
      frame_paths → letter sequence → word candidates → grammar check → coherence
    """
    # Step 1: CNN letter classification
    letters, confidences = [], []
    for path in frame_paths:
        letter, conf = predict_letter_from_image(path)
        if letter not in ('del', 'space', 'nothing'):  # skip control classes
            letters.append(letter)
            confidences.append(conf)

    letter_seq = ''.join(letters)
    mean_conf  = float(np.mean(confidences)) if confidences else 0.0

    # Step 2: n-gram word completion
    word_candidates = complete_word(letter_seq, top_k=5)
    best_word = word_candidates[0] if word_candidates else letter_seq.lower()

    # Step 3: Grammar check
    grammar_result = grammar_check(best_word)

    # Step 4: Coherence (distilBERT)
    coherence_result = predict_coherence(best_word)

    result = {
        'letter_sequence':   letter_seq,
        'mean_confidence':   mean_conf,
        'word_candidates':   word_candidates,
        'best_word':         best_word,
        'grammar_errors':    grammar_result['n_errors'],
        'corrected':         grammar_result['corrected'],
        'coherence_label':   coherence_result['label'],
        'coherence_prob':    coherence_result['prob_coherent'],
    }

    if verbose:
        print('=' * 55)
        print(f'  Letter sequence  : {letter_seq}')
        print(f'  Mean CNN conf    : {mean_conf*100:.1f}%')
        print(f'  Word candidates  : {word_candidates}')
        print(f'  Best word        : {best_word}')
        print(f'  Grammar errors   : {grammar_result["n_errors"]}')
        print(f'  Corrected        : {grammar_result["corrected"]}')
        print(f'  Coherence        : {coherence_result["label"]} ({coherence_result["prob_coherent"]*100:.1f}%)')
        print('=' * 55)

    return result


# ── Demo with real images from the dataset ────────────────────────────────
# Grab sample frames that spell 'H-E-L-L-O'
demo_letters = ['H', 'E', 'L', 'L', 'O']
demo_paths   = []
for letter in demo_letters:
    folder = os.path.join(TRAIN_DIR, letter)
    if os.path.isdir(folder):
        imgs = glob.glob(os.path.join(folder, '*.jpg'))[:1]
        demo_paths.extend(imgs)

if demo_paths:
    print(f'Demo pipeline — spelling: {demo_letters}')
    result = run_full_pipeline(demo_paths)
else:
    print('Demo paths not found — using simulated input')
    result = {
        'letter_sequence': 'HELLO',
        'word_candidates': complete_word('HELLO'),
        'best_word': 'hello',
        'grammar_errors': 0,
        'corrected': 'hello',
        'coherence_label': predict_coherence('hello')['label'],
        'coherence_prob':  predict_coherence('hello')['prob_coherent'],
    }
    print(result)

---
## Cell 22 — Pipeline Demo Visualization

In [ ]:
# ── Show the frame sequence + predictions side by side ────────────────────
if demo_paths:
    n_frames = len(demo_paths)
    fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3.5))
    if n_frames == 1:
        axes = [axes]

    for ax, path in zip(axes, demo_paths):
        letter, conf = predict_letter_from_image(path)
        img = Image.open(path).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{letter}\n{conf*100:.1f}%', fontsize=12,
                     color='green' if conf > 0.7 else 'orange')

    plt.suptitle(f'Gesture Frames → Letters → Word: "{result["best_word"]}"\n'
                 f'Coherence: {result["coherence_label"]} ({result["coherence_prob"]*100:.1f}%)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/pipeline_demo.png', dpi=150)
    plt.show()
    print('Saved: pipeline_demo.png')
else:
    print('Skipped (no demo images found — re-run Cell 21 after confirming TRAIN_DIR)')

---
## Cell 23 — Real-Time Webcam Inference (Optional, runs in Colab)

Captures a photo from your webcam, runs it through the full pipeline.

In [ ]:
# ── Webcam capture in Colab ────────────────────────────────────────────────
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import PIL

def take_photo(filename='webcam.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '📸 Capture Sign';
      capture.style = 'font-size:18px;padding:8px 20px;margin:10px;cursor:pointer';
      div.appendChild(capture);
      document.body.appendChild(div);

      const video = document.createElement('video');
      video.style = 'display:block;margin:10px auto;border:2px solid #333;border-radius:8px';
      video.setAttribute('playsinline', '');
      div.appendChild(video);

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(video);
      video.srcObject = stream;
      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width  = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(t => t.stop());
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename


def preprocess_webcam(img_path: str):
    """
    Preprocesses a webcam capture:
    - Convert BGR→RGB (if from OpenCV)
    - Add 20% padding around the center crop
    - Resize to IMG_SIZE, apply val_transform
    """
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    # Center crop
    side  = min(w, h)
    left  = (w - side) // 2
    upper = (h - side) // 2
    img   = img.crop((left, upper, left + side, upper + side))
    return img


# ── Capture and predict ────────────────────────────────────────────────────
print('Click "📸 Capture Sign" to take a photo, then we will predict the letter.')
try:
    photo_path = take_photo('/content/webcam_sign.jpg')
    img = preprocess_webcam(photo_path)
    img.save('/content/webcam_preprocessed.jpg')

    letter, conf = predict_letter_from_image('/content/webcam_preprocessed.jpg')

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(Image.open(photo_path))
    axes[0].set_title('Captured Frame', fontsize=11)
    axes[0].axis('off')
    axes[1].imshow(img)
    axes[1].set_title(f'Predicted: {letter} ({conf*100:.1f}%)',
                      fontsize=13, fontweight='bold',
                      color='green' if conf > 0.7 else 'orange')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('/content/webcam_result.png', dpi=150)
    plt.show()
    print(f'Predicted letter: {letter}  |  Confidence: {conf*100:.1f}%')
except Exception as e:
    print(f'Webcam capture failed (normal in non-browser env): {e}')
    print('→ To test, place an image at /content/webcam_sign.jpg and rerun the prediction block.')

---
## Cell 24 — Save All Results & Summary

In [ ]:
import json

# ── Save training history ─────────────────────────────────────────────────
with open('/content/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

# ── Save per-class accuracy ───────────────────────────────────────────────
per_class_dict = {class_labels[i]: float(per_class_acc[i]) for i in range(NUM_CLASSES)}
with open('/content/per_class_accuracy.json', 'w') as f:
    json.dump(per_class_dict, f, indent=2)

# ── Print final summary ───────────────────────────────────────────────────
print('=' * 60)
print('         P07 — ASL Sign Language Recognition')
print('                   FINAL SUMMARY')
print('=' * 60)
print(f'  Dataset         : ASL Alphabet ({total:,} images, {NUM_CLASSES} classes)')
print(f'  Architecture    : ASLCNN (4 ConvBlocks + GAP + FC)')
print(f'  Parameters      : {train_p:,}')
print(f'  Epochs trained  : {NUM_EPOCHS}')
print(f'  Best val acc    : {best_val_acc*100:.2f}%')
print(f'  CER             : {total_cer*100:.2f}%')
print(f'  WER             : {total_wer*100:.2f}%')
print('  NLP components  : n-gram LM + LanguageTool + distilBERT')
print('=' * 60)
print('\nSaved files in /content/:')
for fname in [
    'best_asl_cnn.pth', 'training_history.json', 'per_class_accuracy.json',
    'training_curves.png', 'confusion_matrix.png', 'per_class_accuracy.png',
    'misclassified.png', 'gradcam.png', 'cer_wer.png',
    'class_distribution.png', 'sample_images.png', 'pipeline_demo.png',
]:
    path = f'/content/{fname}'
    exists = '✅' if os.path.exists(path) else '❌'
    print(f'  {exists} {fname}')

---
## Cell 25 — Download All Outputs as ZIP

In [ ]:
import shutil
from google.colab import files

# ── Bundle all outputs ─────────────────────────────────────────────────────
os.makedirs('/content/P07_outputs', exist_ok=True)
output_files = [
    'best_asl_cnn.pth', 'training_history.json', 'per_class_accuracy.json',
    'training_curves.png', 'confusion_matrix.png', 'per_class_accuracy.png',
    'misclassified.png', 'gradcam.png', 'cer_wer.png',
    'class_distribution.png', 'sample_images.png', 'pipeline_demo.png',
]
for fname in output_files:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/P07_outputs/{fname}')

shutil.make_archive('/content/P07_ASL_Results', 'zip', '/content/P07_outputs')
print('Downloading P07_ASL_Results.zip ...')
files.download('/content/P07_ASL_Results.zip')